In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from safetensors.torch import load_file

if "../src" not in sys.path:
    sys.path.append("../src")

from cinic10.models.factory import replace_conv2d_with_convkan
from cinic10.models.nas_cnn import DiscreteNasCnn

SEEDS = [0, 42, 3407]

Logging to file: /Users/pawelflorek/PW/DS/sem1/DL/cinic10/logs/cinic10.log


In [2]:
summary = pd.DataFrame()

for seed in SEEDS:
    densenet = (
        pd.read_json(
            f"../outputs/03_models/densenet121_finetune/seed_{seed}/test_metrics.json",
            typ="series",
        )
        .to_frame()
        .T
    )
    resnet = (
        pd.read_json(
            f"../outputs/03_models/resnet18_finetune/seed_{seed}/test_metrics.json",
            typ="series",
        )
        .to_frame()
        .T
    )
    nas = (
        pd.read_json(
            f"../outputs/03_models/nas_two_stage/seed_{seed}/two_stage_summary.json",
            typ="series",
        )
        .to_frame()
        .T[["test_accuracy", "test_loss"]]
    )
    squeeznet = (
        pd.read_json(
            f"../outputs/03_models/squeezenet/seed_{seed}/test_metrics.json",
            typ="series",
        )
        .to_frame()
        .T
    )
    mobilenet = (
        pd.read_json(
            f"../outputs/02_aug/mobilenet_standard/seed_{seed}/test_metrics.json",
            typ="series",
        )
        .to_frame()
        .T
    )

    summary = pd.concat(
        [
            summary,
            pd.DataFrame(
                [
                    {
                        "seed": seed,
                        "densenet_test_accuracy": densenet["test_accuracy"].values[0],
                        "resnet_test_accuracy": resnet["test_accuracy"].values[0],
                        "nas_test_accuracy": nas["test_accuracy"].values[0],
                        "squeezenet_test_accuracy": squeeznet["test_accuracy"].values[0],
                        "mobilenet_test_accuracy": mobilenet["test_accuracy"].values[0],
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

In [3]:
summary.drop("seed", axis=1).agg(["mean", "std"]).T.sort_values("mean", ascending=False)

,mean,std
densenet_test_accuracy,0.759952,0.001201
resnet_test_accuracy,0.723237,0.003234
squeezenet_test_accuracy,0.586433,0.000385
mobilenet_test_accuracy,0.557856,0.005937
nas_test_accuracy,0.551293,0.035654


# NAS analysis

In [4]:
nas_root = Path("../outputs/03_models/nas_two_stage")

nas_rows = []

for seed in SEEDS:
    seed_dir = nas_root / f"seed_{seed}"
    summary_path = seed_dir / "two_stage_summary.json"
    summary_series = pd.read_json(summary_path, typ="series")

    search = summary_series["search"]
    retrain = summary_series["retrain"]
    selected_ops = summary_series["selected_operation_names"]

    # Main NAS metrics per seed.
    nas_rows.append(
        {
            "seed": seed,
            "search_best_val_accuracy": search["best_val_accuracy"],
            "search_best_val_loss": search["best_val_loss"],
            "retrain_best_val_accuracy": retrain["best_val_accuracy"],
            "retrain_best_val_loss": retrain["best_val_loss"],
            "test_accuracy": summary_series["test_accuracy"],
            "test_loss": summary_series["test_loss"],
            "search_to_retrain_gap": search["best_val_accuracy"] - retrain["best_val_accuracy"],
            "selected_ops": " | ".join(selected_ops),
        }
    )

nas_summary = (
    pd.DataFrame(nas_rows).sort_values("test_accuracy", ascending=False).reset_index(drop=True)
)

display(
    nas_summary[
        [
            "seed",
            "search_best_val_accuracy",
            "retrain_best_val_accuracy",
            "test_accuracy",
        ]
    ]
)
display(
    nas_summary[
        [
            "search_best_val_accuracy",
            "retrain_best_val_accuracy",
            "test_accuracy",
            "search_to_retrain_gap",
        ]
    ]
    .agg(["mean", "std"])
    .T
)

,seed,search_best_val_accuracy,retrain_best_val_accuracy,test_accuracy
0,3407,0.684844,0.590056,0.587967
1,42,0.667756,0.549189,0.549156
2,0,0.653911,0.515178,0.516756


,mean,std
search_best_val_accuracy,0.668837,0.015495
retrain_best_val_accuracy,0.551474,0.037491
test_accuracy,0.551293,0.035654
search_to_retrain_gap,0.117363,0.021997


In [2]:
seed = 3407
nas_seed_dir = Path(f"../outputs/03_models/nas_two_stage/seed_{seed}")
architecture_path = nas_seed_dir / "search" / "architecture.json"
weights_path = nas_seed_dir / "retrain" / "best.safetensors"

arch_payload = json.loads(architecture_path.read_text(encoding="utf-8"))
selected_ops = arch_payload["selected_operation_names"]

standard_model = DiscreteNasCnn(selected_operations=selected_ops, num_classes=10, dropout=0.1)
state_dict = load_file(str(weights_path))
standard_model.load_state_dict(state_dict, strict=False)
standard_model.eval()

DiscreteNasCnn(
  (backbone): Sequential(
    (0): Sequential(
      (0): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
      (1): _Projection(
        (net): Sequential(
          (0): Conv2d(3, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
    )
    (1): Sequential(
      (0): Conv2d(32, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(5, 5), stride=(2, 2), padding=(2, 2), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (3): _DepthwiseSeparableConv(
      (net): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(

In [3]:
nas_convkan_model = replace_conv2d_with_convkan(standard_model)
nas_convkan_model.eval()

DiscreteNasCnn(
  (backbone): Sequential(
    (0): Sequential(
      (0): MaxPool2d(kernel_size=3, stride=1, padding=1, dilation=1, ceil_mode=False)
      (1): _Projection(
        (net): Sequential(
          (0): Conv2d(3, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
      )
    )
    (1): Sequential(
      (0): KANConvolutionalLayer(
        (convs): ModuleList(
          (0-1023): 1024 x KANConvolution(
            (conv): KANLinear(
              (base_activation): SiLU()
            )
          )
        )
      )
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (2): Sequential(
      (0): KANConvolutionalLayer(
        (convs): ModuleList(
          (0-2047): 2048 x KANConvolution(
            (conv): KANLinear(
              (base_activation): SiLU()
            )
          )
        )
  

In [4]:
import time

import torch

# Minimal smoke test for ConvKAN-converted NAS model.
test_batch = torch.randn(1, 3, 32, 32)
start = time.perf_counter()

with torch.no_grad():
    logits = nas_convkan_model(test_batch)
    probs = torch.softmax(logits, dim=1)

elapsed = time.perf_counter() - start

print("model:", nas_convkan_model.__class__.__name__)
print("input shape:", tuple(test_batch.shape))
print("logits shape:", tuple(logits.shape))
print("all finite logits:", torch.isfinite(logits).all().item())
print("prob sum:", float(probs.sum(dim=1).item()))
print("pred class:", int(logits.argmax(dim=1).item()))
print("forward seconds:", round(elapsed, 4))

model: DiscreteNasCnn
input shape: (1, 3, 32, 32)
logits shape: (1, 10)
all finite logits: True
prob sum: 1.0000001192092896
pred class: 5
forward seconds: 4.2544
